Install the dependencies
pip install langchain langchain-ollama langchain-community

In [1]:
#Importing required libraries
import chromadb
from sentence_transformers import SentenceTransformer
from langchain_ollama import OllamaLLM

In [2]:
# ─────────────────────────────────────────
# 1. SETUP — embedding model + LLM + vector DB
# ─────────────────────────────────────────

embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

# Connect to local Ollama LLM
llm = OllamaLLM(model="llama3.2:3b")

# Local ChromaDB (in-memory)
client = chromadb.Client()
collection = client.create_collection("company_docs")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [3]:
# ─────────────────────────────────────────
# 2. INDEXING — your "documents" (fake company data)
# ─────────────────────────────────────────

documents = [
    "Q3 revenue was $2.3M, up 12% year over year",
    "Product A led sales with 40% market share",
    "Customer churn decreased to 5% this quarter",
    "The engineering team shipped 3 major features in Q3",
    "Net Promoter Score improved from 32 to 47 this quarter",
    "Product B underperformed, missing targets by 18%",
]

# Embed all documents and store in ChromaDB
embeddings = embedding_model.encode(documents).tolist()

collection.add(
    documents=documents,
    embeddings=embeddings,
    ids=[f"doc{i}" for i in range(len(documents))]
)

print("✅ Documents indexed\n")

✅ Documents indexed



In [4]:
# ─────────────────────────────────────────
# 3. RAG FUNCTION — retrieve + generate
# ─────────────────────────────────────────

def ask(question):
    # Step A: embed the question
    query_embedding = embedding_model.encode(question).tolist()

    # Step B: find top 3 most relevant chunks from ChromaDB
    results = collection.query(
        query_embeddings=[query_embedding],
        n_results=3
    )
    retrieved_chunks = results["documents"][0]

    # Step C: build the prompt — inject chunks as context
    context = "\n".join(f"- {chunk}" for chunk in retrieved_chunks)

    prompt = f"""You are a helpful data analyst assistant.
Answer the question using ONLY the context provided below.
If the answer is not in the context, say "I don't have that information."

Context:
{context}

Question: {question}

Answer:"""

    # Step D: send to LLM and return answer
    print(f"📋 Retrieved context:\n{context}\n")
    print(f"🤖 Answer:")
    response = llm.invoke(prompt)
    print(response)
    print("\n" + "─"*50 + "\n")


In [5]:
# ─────────────────────────────────────────
# 4. ASK QUESTIONS Q1
# ─────────────────────────────────────────

ask("How did the company perform in Q3?")

📋 Retrieved context:
- The engineering team shipped 3 major features in Q3
- Q3 revenue was $2.3M, up 12% year over year
- Product A led sales with 40% market share

🤖 Answer:
The company performed well in Q3, with a 12% increase in revenue compared to the same quarter last year. The engineering team successfully shipped three major features, and product A continued to drive sales with a significant 40% market share.

──────────────────────────────────────────────────



In [6]:
#Question 2
ask("Which products did well and which didn't?")

📋 Retrieved context:
- Product B underperformed, missing targets by 18%
- Product A led sales with 40% market share
- Q3 revenue was $2.3M, up 12% year over year

🤖 Answer:
Based on the context, I can conclude that:

Product B underperformed, so it didn't do well.
Product A led sales with a 40% market share, indicating it did very well.

I don't have information about Product C.

──────────────────────────────────────────────────



In [7]:
#Question 3
ask("What is the weather like in Paris?")   

📋 Retrieved context:
- The engineering team shipped 3 major features in Q3
- Product B underperformed, missing targets by 18%
- Product A led sales with 40% market share

🤖 Answer:
I don't have that information.

──────────────────────────────────────────────────



Your question
    ↓
Embed question → query vector
    ↓
ChromaDB: find top 3 similar chunks
    ↓
Build prompt:
  "Here is context: [chunk1] [chunk2] [chunk3]
   Answer this: {question}"
    ↓
Ollama/Llama3.2 reads prompt → generates answer
    ↓
Answer is grounded in YOUR data, not model's training